In [1]:
import sqlite3
import pandas as pd

# 1. LOAD CLEANED CSVs INTO SQLITE DATABASE


In [2]:
print("Connecting to SQLite database and loading tables...")

# Create/Connect to local database file
conn = sqlite3.connect("sales_analysis.db")

# Load cleaned transactional data
df_transactions = pd.read_csv("cleaned_online_retail.csv")
df_transactions.to_sql("transactions", conn, if_exists="replace", index=False)

# Load RFM customer segments data
df_rfm = pd.read_csv("rfm_customer_segments.csv")
df_rfm.to_sql("rfm_segments", conn, if_exists="replace", index=False)

print("Tables 'transactions' and 'rfm_segments' successfully created in SQLite!\n")

Connecting to SQLite database and loading tables...
Tables 'transactions' and 'rfm_segments' successfully created in SQLite!



# SQL QUERY 1: REVENUE AT RISK BY CUSTOMER SEGMENT (JOIN + CTE)

In [3]:
print("--- QUERY 1: Revenue at Risk by Customer Segment ---")

query_1 = """
WITH SegmentSummary AS (
    SELECT 
        Segment,
        COUNT(CustomerID) AS TotalCustomers,
        SUM(Monetary) AS SegmentRevenue,
        SUM(CASE WHEN Is_At_Risk = 1 THEN Monetary ELSE 0 END) AS AtRiskRevenue
    FROM rfm_segments
    GROUP BY Segment
)
SELECT 
    Segment,
    TotalCustomers,
    ROUND(SegmentRevenue, 2) AS SegmentRevenue,
    ROUND(AtRiskRevenue, 2) AS AtRiskRevenue,
    ROUND((AtRiskRevenue / SegmentRevenue) * 100, 2) AS Pct_At_Risk
FROM SegmentSummary
ORDER BY AtRiskRevenue DESC;
"""

df_q1 = pd.read_sql_query(query_1, conn)
print(df_q1.to_string(index=False))
print("\n" + "="*70 + "\n")

--- QUERY 1: Revenue at Risk by Customer Segment ---
           Segment  TotalCustomers  SegmentRevenue  AtRiskRevenue  Pct_At_Risk
           At Risk             343       526713.40      431381.66        81.90
Hibernating / Lost            1065       519408.57      221930.20        42.73
   Needs Attention             275       220751.63      121193.66        54.90
    Cant Lose Them              25        53066.52       53066.52       100.00
         Champions             962      5809341.07           0.00         0.00
   Loyal Customers             998      1474145.55           0.00         0.00
Recent / Promising             670       307999.16           0.00         0.00




# SQL QUERY 2: TOP 10 AT-RISK CUSTOMERS BY REVENUE (JOIN + CTE)

In [4]:
print("--- QUERY 2: Top 10 High-Value At-Risk Customers ---")

query_2 = """
WITH AtRiskList AS (
    SELECT 
        CustomerID,
        Recency,
        Frequency,
        Monetary,
        Segment
    FROM rfm_segments
    WHERE Is_At_Risk = 1
)
SELECT 
    r.CustomerID,
    r.Segment,
    r.Recency AS Days_Since_Last_Order,
    r.Frequency AS Total_Orders,
    ROUND(r.Monetary, 2) AS Total_Lifetime_Spend
FROM AtRiskList r
ORDER BY r.Monetary DESC
LIMIT 10;
"""

df_q2 = pd.read_sql_query(query_2, conn)
print(df_q2.to_string(index=False))
print("\n" + "="*70 + "\n")

--- QUERY 2: Top 10 High-Value At-Risk Customers ---
 CustomerID            Segment  Days_Since_Last_Order  Total_Orders  Total_Lifetime_Spend
      12346 Hibernating / Lost                    326             1              77183.60
      15749    Needs Attention                    235             3              44534.30
      15098    Needs Attention                    182             3              39916.50
      12409            At Risk                     79             3              11072.67
      16180            At Risk                    100             8              10254.18
      12590 Hibernating / Lost                    211             2               9864.26
      13093     Cant Lose Them                    276             8               7832.47
      12435 Hibernating / Lost                     80             2               7829.89
      12980            At Risk                    158             9               7374.90
      16745            At Risk                 

# SQL QUERY 3: PURCHASE GAPS USING WINDOW FUNCTIONS (LAG)

In [5]:
print("--- QUERY 3: Average Order Gap (Days Between Purchases) ---")

query_3 = """
WITH OrderDates AS (
    SELECT DISTINCT
        CustomerID,
        DATE(InvoiceDate) AS OrderDate
    FROM transactions
),
OrderGaps AS (
    SELECT 
        CustomerID,
        OrderDate,
        LAG(OrderDate, 1) OVER (
            PARTITION BY CustomerID 
            ORDER BY OrderDate
        ) AS PriorOrderDate
    FROM OrderDates
),
CalculatedGaps AS (
    SELECT 
        CustomerID,
        JULIANDAY(OrderDate) - JULIANDAY(PriorOrderDate) AS DaysBetweenOrders
    FROM OrderGaps
    WHERE PriorOrderDate IS NOT NULL
)
SELECT 
    r.Segment,
    ROUND(AVG(g.DaysBetweenOrders), 1) AS Avg_Days_Between_Orders
FROM CalculatedGaps g
JOIN rfm_segments r ON g.CustomerID = r.CustomerID
GROUP BY r.Segment
ORDER BY Avg_Days_Between_Orders ASC;
"""

df_q3 = pd.read_sql_query(query_3, conn)
print(df_q3.to_string(index=False))
print("\n" + "="*70 + "\n")

--- QUERY 3: Average Order Gap (Days Between Purchases) ---
           Segment  Avg_Days_Between_Orders
         Champions                     32.2
    Cant Lose Them                     34.3
           At Risk                     62.1
   Needs Attention                     69.0
   Loyal Customers                     75.4
Hibernating / Lost                     99.2
Recent / Promising                    136.9




# SQL QUERY 4: COHORT RETENTION BY SIGNUP MONTH (JOIN + CTE)

In [6]:
print("--- QUERY 4: Cohort Revenue Retention (First Month vs Later) ---")

query_4 = """
WITH FirstPurchase AS (
    SELECT 
        CustomerID,
        STRFTIME('%Y-%m', MIN(InvoiceDate)) AS CohortMonth
    FROM transactions
    GROUP BY CustomerID
),
MonthlySpend AS (
    SELECT 
        t.CustomerID,
        fp.CohortMonth,
        STRFTIME('%Y-%m', t.InvoiceDate) AS OrderMonth,
        SUM(t.TotalLineAmount) AS MonthlyRevenue
    FROM transactions t
    JOIN FirstPurchase fp ON t.CustomerID = fp.CustomerID
    GROUP BY t.CustomerID, fp.CohortMonth, OrderMonth
)
SELECT 
    CohortMonth,
    COUNT(DISTINCT CustomerID) AS CohortSize,
    ROUND(SUM(CASE WHEN CohortMonth = OrderMonth THEN MonthlyRevenue ELSE 0 END), 2) AS Initial_Month_Revenue,
    ROUND(SUM(CASE WHEN CohortMonth != OrderMonth THEN MonthlyRevenue ELSE 0 END), 2) AS Retention_Revenue
FROM MonthlySpend
GROUP BY CohortMonth
ORDER BY CohortMonth ASC
LIMIT 6;
"""

df_q4 = pd.read_sql_query(query_4, conn)
print(df_q4.to_string(index=False))

--- QUERY 4: Cohort Revenue Retention (First Month vs Later) ---
CohortMonth  CohortSize  Initial_Month_Revenue  Retention_Revenue
    2010-12         885              572713.89         3939434.33
    2011-01         417              293207.35          832674.93
    2011-02         380              158142.07          435734.81
    2011-03         452              200069.96          443688.98
    2011-04         300              122011.49          204609.54
    2011-05         284              124103.78          331441.60


In [7]:
# Close connection
conn.close()